# Notebook 02 — Popularity Analysis
**Feature 1.7 — EDA & Data Understanding | HitRadar Pro**

## Mục tiêu
- Phân tích phân bố `target_popularity` — đây là **label ML**, KHÔNG phải input feature.
- Tìm hiểu popularity theo bucket, theo thập kỷ, và xác định tình trạng mất cân bằng.
- Insight này phục vụ quyết định xử lý imbalance ở EPIC 2.

⚠️ **Lưu ý:** `target_popularity = clean.tracks.popularity` — là biến mục tiêu dự đoán.

In [ ]:
import os, psycopg2, pandas as pd, matplotlib.pyplot as plt
import matplotlib.ticker as mticker

conn = psycopg2.connect(
    host='localhost', port=5432, user='postgres',
    password=os.environ.get('PGPASSWORD', 'hitradar123'), dbname='hitradar'
)
print('Kết nối thành công.')

## 1. Phân bố Popularity theo Bucket

In [ ]:
df_buckets = pd.read_sql("""
    SELECT popularity_bucket, track_count, min_popularity, max_popularity,
           avg_popularity, median_popularity
    FROM analytics.vw_popularity_stats
    ORDER BY min_popularity
""", conn)

df_buckets['pct'] = df_buckets['track_count'] / df_buckets['track_count'].sum() * 100
df_buckets[['popularity_bucket','track_count','pct','avg_popularity','median_popularity']]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Biểu đồ 1: số tracks theo bucket
colors = ['#d62728','#ff7f0e','#2ca02c','#1f77b4','#9467bd']
axes[0].bar(df_buckets['popularity_bucket'], df_buckets['track_count'], color=colors)
axes[0].set_title('Số tracks theo khoảng popularity', fontweight='bold')
axes[0].set_xlabel('Popularity bucket')
axes[0].set_ylabel('Số tracks')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Biểu đồ 2: phần trăm
axes[1].bar(df_buckets['popularity_bucket'], df_buckets['pct'], color=colors)
axes[1].set_title('Tỷ lệ % theo khoảng popularity', fontweight='bold')
axes[1].set_xlabel('Popularity bucket')
axes[1].set_ylabel('%')
for i, (_, row) in enumerate(df_buckets.iterrows()):
    axes[1].text(i, row['pct'] + 0.5, f"{row['pct']:.1f}%", ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 2. Popularity theo Thập kỷ

In [ ]:
df_decade = pd.read_sql("""
    SELECT decade, track_count, avg_popularity, median_popularity
    FROM analytics.vw_tracks_by_decade
    WHERE decade >= 1920
    ORDER BY decade
""", conn)
df_decade

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
x = df_decade['decade'].astype(str)
ax.plot(x, df_decade['avg_popularity'], marker='o', color='steelblue', label='Avg popularity')
ax.plot(x, df_decade['median_popularity'], marker='s', linestyle='--',
        color='darkorange', label='Median popularity')
ax.set_title('Popularity trung bình và trung vị theo thập kỷ', fontweight='bold')
ax.set_xlabel('Thập kỷ')
ax.set_ylabel('Popularity (0–100)')
ax.legend()
ax.set_ylim(0, 55)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Phân bố chi tiết target_popularity

In [ ]:
df_pop_dist = pd.read_sql("""
    SELECT target_popularity AS popularity,
           COUNT(*) AS cnt
    FROM analytics.vw_ml_training_dataset
    GROUP BY target_popularity
    ORDER BY target_popularity
""", conn)

zero_tracks = df_pop_dist.loc[df_pop_dist.popularity == 0, 'cnt'].values[0]
print(f'Tracks có popularity = 0: {zero_tracks:,} ({zero_tracks/586672*100:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df_pop_dist['popularity'], df_pop_dist['cnt'], width=1.0, color='steelblue', alpha=0.8)
ax.set_title('Phân bố target_popularity (0–100)  |  ⚠️ Đây là LABEL — không dùng làm input feature',
             fontweight='bold')
ax.set_xlabel('target_popularity')
ax.set_ylabel('Số tracks')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()

## 4. Nhận xét & Kết luận

**Phân bố:**
- **Rất lệch về thấp (right-skewed / heavy low tail):**
  - Bucket 0–20: **37.5%** (219,988 tracks) — nhóm lớn nhất
  - Bucket 21–40: **37.3%** (219,003 tracks)
  - Bucket 41–60: **20.9%** (122,813 tracks)
  - Bucket 61–80: **4.1%** (24,132 tracks)
  - Bucket 81–100: **chỉ 0.1%** (736 tracks)
- **44,690 tracks** có `popularity = 0` (~7.6%) — cần xử lý ở EPIC 2 (có thể là inactive/unlisted tracks).

**Theo thập kỷ:**
- Popularity tăng dần theo thời gian: 1920s avg ≈ 1 → 2020s avg ≈ 42.
- Điều này chủ yếu do **Spotify tính popularity dựa trên streams gần đây** — tracks mới có lợi thế.
- `release_year` sẽ có tương quan cao với target (sẽ xác nhận ở Notebook 06).

**Lưu ý cho EPIC 2:**
- Class imbalance nghiêm trọng — cần xem xét log-transform, phân lớp popularity, hoặc sample strategy.
- `target_popularity` là label — **KHÔNG dùng làm input feature**.

In [ ]:
conn.close()
print('Done — Notebook 02 hoàn thành.')